In [61]:
import pandas as pd
import numpy as np

original_data_path = "../data/raw/"
data_binary_representation ="../data/processed/"

In [62]:
df = pd.read_csv(original_data_path + "wells.csv")
df_concentration = pd.read_csv(original_data_path + "concentration.csv")
df_head = pd.read_csv(original_data_path + "head.csv")
df_cores = pd.read_csv(original_data_path + "cores.csv")
df_pumping = pd.read_csv(original_data_path + "pumping.csv")

Сначала удалим все nan значение, затем удалим каскадно из других таблиц, после удалим дубликаты с округлением по x и y до целых и восстановим z.

In [63]:
df.loc[df['x'] == -999.0, "x"] = np.nan

nan_values_mask = df[['x','y','z']].isna().any(axis=1)

df_deleted = df[nan_values_mask]
df_deleted = df_deleted["well_id"].unique()
df = df[~nan_values_mask]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)

duplicated_mask = df.duplicated(subset=["round_x", "round_y", "z"], keep="first")
duplicated_idx = df[duplicated_mask]["well_id"].unique()

df = df[~duplicated_mask]
df = df.drop(columns=["round_x", "round_y"])

In [64]:
df_concentration = df_concentration[~df_concentration["well_id"].isin(df_deleted)]
df_head = df_head[~df_head["well_id"].isin(df_deleted)]
df_cores = df_cores[~df_cores["well_id"].isin(df_deleted)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(df_deleted)]

df_concentration = df_concentration[~df_concentration["well_id"].isin(duplicated_idx)]
df_head = df_head[~df_head["well_id"].isin(duplicated_idx)]
df_cores = df_cores[~df_cores["well_id"].isin(duplicated_idx)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(duplicated_idx)]

In [65]:
df.loc[(df["z"] > 5) & (df["z"] < 150), "z"] = -df["z"]

In [66]:
df.to_parquet(f"{data_binary_representation}wells.parquet", engine='pyarrow')